# Esteira Geo — Utilitários

Funções de manutenção do ambiente. Execute a célula de configuração antes de qualquer outra.

| Seção | O que faz |
|-------|-----------|
| 0. Configuração | Conecta ao S3/MinIO e PostGIS |
| 1. Limpar banco de dados | Remove tabelas de use_cases do PostGIS |
| 2. Limpar silver e gold | Remove objetos dos buckets silver e gold |
| 3. Mover processados → ativo | Devolve arquivos de `processados/` para reprocessamento |
| 4. Apagar use_case completo | Remove tudo (PostGIS + silver + gold + bronze) de um use_case |

## 0. Configuração

In [1]:
import os, sys
sys.path.insert(0, '/app/pipeline_src')
sys.path.insert(0, '/app')

import importlib, config
importlib.reload(config)

import boto3
import psycopg2

def _s3():
    return boto3.client(
        's3',
        endpoint_url=config.AWS_ENDPOINT_URL,
        aws_access_key_id=config.AWS_ACCESS_KEY_ID,
        aws_secret_access_key=config.AWS_SECRET_ACCESS_KEY,
        region_name=config.AWS_S3_REGION_NAME,
    )

def _conn():
    return psycopg2.connect(
        host=config.RDS_HOST, port=config.RDS_PORT,
        dbname=config.RDS_DATABASE, user=config.RDS_USER,
        password=config.RDS_PASSWORD
    )

def _list_bucket(bucket, prefix=''):
    s3 = _s3()
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    return [o['Key'] for o in resp.get('Contents', [])]

def _delete_keys(bucket, keys):
    if not keys:
        return 0
    _s3().delete_objects(Bucket=bucket, Delete={'Objects': [{'Key': k} for k in keys]})
    return len(keys)

def _use_case_tables(conn, use_case):
    cur = conn.cursor()
    cur.execute("""
        SELECT tablename FROM pg_tables
        WHERE schemaname = 'public' AND tablename LIKE %s
    """, (f'{use_case}_%',))
    tables = [r[0] for r in cur.fetchall()]
    cur.close()
    return tables

print('✓ Configuração carregada')
print(f'  MinIO  : {config.AWS_ENDPOINT_URL}')
print(f'  PostGIS: {config.RDS_HOST}:{config.RDS_PORT}/{config.RDS_DATABASE}')
print(f'  Buckets: {config.AWS_S3_BRONZE_BUCKET} / {config.AWS_S3_SILVER_BUCKET} / {config.AWS_S3_GOLD_BUCKET}')

✓ Configuration loaded (Mode: MINIO)
  Storage: MINIO
    MinIO: http://minio:9000
    Buckets: bronze/enchentes_poa, silver/enchentes_poa, gold/enchentes_poa
  Use Case: enchentes_poa
  Database: postgis:5432/esteira_geo
  Logging: logs/pipeline.log
✓ Configuration loaded (Mode: MINIO)
  Storage: MINIO
    MinIO: http://minio:9000
    Buckets: bronze/enchentes_poa, silver/enchentes_poa, gold/enchentes_poa
  Use Case: enchentes_poa
  Database: postgis:5432/esteira_geo
  Logging: logs/pipeline.log
✓ Configuração carregada
  MinIO  : http://minio:9000
  PostGIS: postgis:5432/esteira_geo
  Buckets: bronze / silver / gold


## 1. Limpar banco de dados

Remove as tabelas de use_cases do PostGIS. As tabelas do Airflow e do PostGIS interno são preservadas.

- `use_case=None` → remove **todos** os use_cases
- `use_case='enchentes_poa'` → remove apenas aquele use_case

In [6]:
def limpar_banco(use_case='enchentes_poa', confirmar=True):
    """
    Remove tabelas de use_case(s) do PostGIS.
    use_case=None remove todos. Requer confirmar=True.
    """
    if not confirmar:
        print('⚠ Operação destrutiva. Chame com confirmar=True para executar.')
        return

    conn = _conn()
    cur = conn.cursor()

    if use_case:
        tables = _use_case_tables(conn, use_case)
    else:
        # Todos os use_cases: tabelas que não são do sistema
        SYSTEM_TABLES = {
            'spatial_ref_sys', 'alembic_version', 'callback_request', 'connection',
            'dag', 'dag_code', 'dag_owner_attributes', 'dag_pickle', 'dag_run',
            'dag_run_note', 'dag_schedule_dataset_reference', 'dag_tag', 'dag_warning',
            'dagrun_dataset_event', 'dataset', 'dataset_dag_run_queue', 'dataset_event',
            'import_error', 'job', 'log', 'log_template', 'rendered_task_instance_fields',
            'serialized_dag', 'session', 'sla_miss', 'slot_pool', 'task_fail',
            'task_instance', 'task_instance_note', 'task_map', 'task_outlet_dataset_reference',
            'task_reschedule', 'trigger', 'variable', 'xcom',
        }
        cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'public'")
        tables = [
            r[0] for r in cur.fetchall()
            if r[0] not in SYSTEM_TABLES and not r[0].startswith('ab_')
        ]

    if not tables:
        print('Nenhuma tabela encontrada.')
        conn.close()
        return

    for table in tables:
        cur.execute(f'DROP TABLE IF EXISTS {table} CASCADE')
        print(f'  ✓ DROP TABLE {table}')

    conn.commit()
    cur.close()
    conn.close()
    print(f'\n✓ {len(tables)} tabela(s) removida(s).')


# Exemplos de uso:
# limpar_banco(confirmar=True)                        # todos os use_cases
# limpar_banco(use_case='enchentes_poa', confirmar=True)  # só enchentes_poa

limpar_banco()  # dry-run — mostra aviso sem executar

  ✓ DROP TABLE enchentes_poa_citizens
  ✓ DROP TABLE enchentes_poa_flooding_areas

✓ 2 tabela(s) removida(s).


## 2. Limpar silver e gold

Remove objetos dos buckets silver e gold. Bronze não é tocado.

- `use_case=None` → limpa **todos** os use_cases
- `use_case='enchentes_poa'` → limpa apenas aquele use_case

In [5]:
def limpar_silver_gold(use_case=None, confirmar=False):
    """
    Remove objetos dos buckets silver e gold.
    use_case=None remove todos. Requer confirmar=True.
    """
    if not confirmar:
        print('⚠ Operação destrutiva. Chame com confirmar=True para executar.')
        return

    prefix = f'{use_case}/' if use_case else ''

    for bucket in [config.AWS_S3_SILVER_BUCKET, config.AWS_S3_GOLD_BUCKET]:
        keys = _list_bucket(bucket, prefix)
        n = _delete_keys(bucket, keys)
        label = f'{use_case}/' if use_case else '(todos)'
        print(f'  ✓ {bucket}/{label}: {n} objeto(s) removido(s)')

    print('\n✓ Silver e gold limpos.')


# Exemplos de uso:
# limpar_silver_gold(confirmar=True)                        # todos
# limpar_silver_gold(use_case='enchentes_poa', confirmar=True)  # só enchentes_poa

limpar_silver_gold()  # dry-run

  ✓ silver/enchentes_poa/: 2 objeto(s) removido(s)
  ✓ gold/enchentes_poa/: 4 objeto(s) removido(s)

✓ Silver e gold limpos.


## 3. Mover processados → ativo

Devolve arquivos de `automatizado/<use_case>/processados/` para `automatizado/<use_case>/`,
permitindo que o watcher ou o Airflow os reprocesse.

- `use_case=None` → move de **todos** os use_cases

In [7]:
def mover_processados(use_case="enchentes_poa", confirmar=True):
    """
    Move arquivos de automatizado/<use_case>/processados/ de volta para automatizado/<use_case>/.
    use_case=None processa todos os use_cases encontrados.
    """
    if not confirmar:
        print('⚠ Chame com confirmar=True para executar.')
        # Mostrar preview do que seria movido
        prefix = f'automatizado/{use_case}/processados/' if use_case else 'automatizado/'
        keys = [k for k in _list_bucket(config.AWS_S3_BRONZE_BUCKET, prefix) if '/processados/' in k]
        if keys:
            print(f'  Seriam movidos {len(keys)} arquivo(s):')
            for k in keys:
                print(f'    {k}')
        else:
            print('  Nenhum arquivo em processados/.')
        return

    s3 = _s3()
    prefix = f'automatizado/{use_case}/processados/' if use_case else 'automatizado/'
    keys = [k for k in _list_bucket(config.AWS_S3_BRONZE_BUCKET, prefix) if '/processados/' in k]

    if not keys:
        print('Nenhum arquivo em processados/.')
        return

    for key in keys:
        new_key = key.replace('/processados/', '/')
        s3.copy_object(
            Bucket=config.AWS_S3_BRONZE_BUCKET,
            CopySource={'Bucket': config.AWS_S3_BRONZE_BUCKET, 'Key': key},
            Key=new_key
        )
        s3.delete_object(Bucket=config.AWS_S3_BRONZE_BUCKET, Key=key)
        print(f'  ✓ {key.split("/")[-1]} → {new_key}')

    print(f'\n✓ {len(keys)} arquivo(s) movido(s) para área ativa.')


# Exemplos de uso:
# mover_processados(confirmar=True)                        # todos os use_cases
# mover_processados(use_case='enchentes_poa', confirmar=True)  # só enchentes_poa

mover_processados()  # dry-run — mostra preview

  ✓ citizens_data.csv → automatizado/enchentes_poa/citizens_data.csv
  ✓ citizens_minas_gerais.csv → automatizado/enchentes_poa/citizens_minas_gerais.csv
  ✓ flooding_areas_porto_alegre.geojson → automatizado/enchentes_poa/flooding_areas_porto_alegre.geojson
  ✓ novos_pontos_a.csv → automatizado/enchentes_poa/novos_pontos_a.csv
  ✓ novos_pontos_b.geojson → automatizado/enchentes_poa/novos_pontos_b.geojson
  ✓ novos_pontos_c.csv → automatizado/enchentes_poa/novos_pontos_c.csv
  ✓ novos_pontos_d.geojson → automatizado/enchentes_poa/novos_pontos_d.geojson
  ✓ teste_jupyter.csv → automatizado/enchentes_poa/teste_jupyter.csv

✓ 8 arquivo(s) movido(s) para área ativa.


## 4. Apagar use_case completo

Remove **tudo** de um use_case específico:
- Tabelas no PostGIS
- Objetos no silver e gold
- Objetos no bronze (área ativa + processados)

> ⚠ Esta operação não pode ser desfeita.

In [9]:
def apagar_use_case(use_case='enchentes_poa', confirmar=True):
    """
    Remove todos os dados de um use_case: PostGIS + silver + gold + bronze.
    Requer confirmar=True.
    """
    if not use_case:
        print('⚠ Informe um use_case específico.')
        return

    if not confirmar:
        print(f'⚠ Apagará TUDO de "{use_case}". Chame com confirmar=True para executar.')
        # Preview
        conn = _conn()
        tables = _use_case_tables(conn, use_case)
        conn.close()
        print(f'  PostGIS : {tables or "(nenhuma tabela)"}')
        for bucket in [config.AWS_S3_SILVER_BUCKET, config.AWS_S3_GOLD_BUCKET, config.AWS_S3_BRONZE_BUCKET]:
            prefix = f'{use_case}/' if bucket != config.AWS_S3_BRONZE_BUCKET else f'automatizado/{use_case}/'
            keys = _list_bucket(bucket, prefix)
            print(f'  {bucket}/{prefix}: {len(keys)} objeto(s)')
        return

    total_removido = 0

    # 1. PostGIS
    conn = _conn()
    cur = conn.cursor()
    tables = _use_case_tables(conn, use_case)
    for table in tables:
        cur.execute(f'DROP TABLE IF EXISTS {table} CASCADE')
        print(f'  ✓ DROP TABLE {table}')
    conn.commit()
    cur.close()
    conn.close()

    # 2. Silver e Gold
    for bucket in [config.AWS_S3_SILVER_BUCKET, config.AWS_S3_GOLD_BUCKET]:
        keys = _list_bucket(bucket, f'{use_case}/')
        n = _delete_keys(bucket, keys)
        total_removido += n
        print(f'  ✓ {bucket}/{use_case}/: {n} objeto(s) removido(s)')

    # 3. Bronze (área ativa + processados)
    keys = _list_bucket(config.AWS_S3_BRONZE_BUCKET, f'automatizado/{use_case}/')
    n = _delete_keys(config.AWS_S3_BRONZE_BUCKET, keys)
    total_removido += n
    print(f'  ✓ bronze/automatizado/{use_case}/: {n} objeto(s) removido(s)')

    print(f'\n✓ use_case "{use_case}" removido ({len(tables)} tabela(s), {total_removido} objeto(s) S3).')


# Exemplos de uso:
# apagar_use_case('enchentes_poa')                    # dry-run — mostra preview
# apagar_use_case('enchentes_poa', confirmar=True)    # executa

apagar_use_case('enchentes_poa')  # dry-run

  ✓ DROP TABLE enchentes_poa_citizens
  ✓ DROP TABLE enchentes_poa_flooding_areas
  ✓ silver/enchentes_poa/: 2 objeto(s) removido(s)
  ✓ gold/enchentes_poa/: 4 objeto(s) removido(s)
  ✓ bronze/automatizado/enchentes_poa/: 7 objeto(s) removido(s)

✓ use_case "enchentes_poa" removido (2 tabela(s), 13 objeto(s) S3).
